# 6 grep and regular expressions

<div class="bp-banner">
  <div class="bp-series">Introduction to the Bash Shell</div>
  <div style="display:flex;align-items:baseline;gap:14px;flex-wrap:wrap;">
    <span class="bp-title">Part II — Pipelines and text extraction</span>
    <span class="bp-meta">Notebook&nbsp;6</span>
  </div>
  <div style="margin-top:10px;max-width:62ch;color:#46506b;">
    Finding lines by pattern, not by literal string — and the pattern language,
    regular expressions, that you will reuse for the rest of the course.
  </div>
  <div class="bp-rule" style="display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px;">
    <span class="bp-meta">Raymond Amador</span>
    <span class="bp-meta">v0.1.0&nbsp;·&nbsp;CC&nbsp;BY&nbsp;4.0 (text) / MIT (code)</span>
  </div>
</div>

In [1]:
# Hidden setup: stand at the repo root, source the validation gate. grep is
# read-only; the few exercises that save output write into a fresh scratch/.
ROOT="$PWD"; while [ ! -f "$ROOT/tools/check.sh" ] && [ "$ROOT" != "/" ]; do ROOT="$(dirname "$ROOT")"; done
source "$ROOT/tools/check.sh"
set +H
cd "$ROOT"

## What this notebook is about

Notebook 5 found *files* by their names. This notebook looks *inside* them, for
**text** — and it does so by **pattern**, not by exact string. The tool is `grep`;
the pattern language is **regular expressions**.

Those two are not equally important. `grep` is one command among many. Regular
expressions are *the* most reusable text skill in this whole course: the same
patterns drive `sed` in Notebook 8 and the Vim search-and-replace from Notebook 9.
Learn them once here and they pay off everywhere. So we teach both — the tool, and
the language behind it.

The files are real CP2K run logs now living in `data/logs/`. Matching `Total
energy` or a warning in one of them needs **no physics** — it is just pattern on
text.

## A. `grep` with literal patterns

At its simplest, `grep PATTERN FILE` prints every line of the file that contains
the pattern. Start literal: find the total-energy lines in a run log.

```{command-card} grep
```

In [2]:
grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log

 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.246533543175843


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.438732194981611


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.878629961526173


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -143.350064790371448


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -144.026841371125641


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -145.725465772527684


Six lines — the energy after each step of the run. Note the **single quotes**
around the pattern: get into that habit now. They stop the shell from touching the
pattern (expanding a `$`, a `*`, and so on) before `grep` ever sees it — exactly
the "the shell goes first" lesson from globbing, and a habit that becomes essential
the moment your patterns contain regex characters.

The workhorse flags turn a match into an answer. Just *count* the matches with
`-c`:

In [3]:
grep -c 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log

6


Number the matches with `-n`, case-fold with `-i` (the log shouts `WARNING` in
capitals; `-i` catches it however it is spelled):

In [4]:
grep -i 'warning' data/logs/gr2hno3-nvt.log

 *** WARNING in motion/simpar_methods.F:223 :: A temperature tolerance ***


 *** WARNING in dbcsr_mm.F:295 :: Using a non-square number of MPI ranks ***


 The number of warnings for this run is : 2


And `-l` answers "**which files** contain this?" without printing the matches —
indispensable across a directory of logs:

In [5]:
grep -l 'Total FORCE_EVAL' data/logs/*.log

data/logs/gr2hno3-nvt.log


data/logs/gr2hno3-restart.log


## B. Regular expressions

A literal pattern is the floor. The ceiling is a **regular expression**: a small
language for describing *shapes* of text — "a line starting with `ENERGY`", "a
signed decimal number", "either `warning` or `error`". We teach the **Extended**
flavour (ERE), which you reach with `grep -E`. Here is the workhorse set.

<div class="bp-card">
  <span class="bp-card-cmd">Regular expressions (ERE)</span> — <span class="bp-card-job">the pattern language, used with <code>grep -E</code> (and later <code>sed -E</code> and Vim). Curated essentials, not a dictionary.</span>
  <table>
    <tr><td>^   $</td><td>anchor to the start / end of the line</td></tr>
    <tr><td>.</td><td>any single character</td></tr>
    <tr><td>[abc] [^abc]</td><td>one character in / not in the set; ranges like <code>[0-9]</code></td></tr>
    <tr><td>[[:digit:]] [[:space:]]</td><td>named POSIX classes (a digit; whitespace)</td></tr>
    <tr><td>* + ?</td><td>the preceding item: zero-or-more / one-or-more / zero-or-one</td></tr>
    <tr><td>{n,m}</td><td>between n and m repetitions</td></tr>
    <tr><td>a|b</td><td>alternation: match a <b>or</b> b</td></tr>
    <tr><td>( … )</td><td>group, e.g. to apply a quantifier or alternation to several characters</td></tr>
    <tr><td>\.</td><td>backslash escapes a metacharacter to mean it literally (a real dot)</td></tr>
  </table>
</div>

Read one slowly, piece by piece — the pattern that matches a signed decimal like
the energies above:

<div style="background:#1b2233;border:1px solid #0e1422;border-radius:8px;padding:8px 20px 16px;margin:18px 0;">
  <div style="font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:.12em;text-transform:uppercase;color:#c0851a;font-weight:600;margin:8px 0 6px;">Anatomy of one pattern: a signed decimal</div>
  <pre style="background:transparent;color:#cdd2db;margin:0;font-family:'JetBrains Mono',monospace;font-size:0.84rem;line-height:1.5;">    -?  [0-9]+  \.  [0-9]+        matches e.g.   -142.246533
    │     │     │     │
    │     │     │     └─ [0-9]+  one or more digits   (the decimals)
    │     │     └─────── \.      a literal dot        (escaped — not "any char")
    │     └───────────── [0-9]+  one or more digits   (before the dot)
    └─────────────────── -?      an optional minus    ( ? = zero or one )</pre>
</div>

Watch it work. Anchored to the line start, "`ENERGY` after any leading spaces"
finds the energy lines (`^`, the POSIX space class, and a quantifier together):

In [6]:
grep -E '^[[:space:]]*ENERGY' data/logs/gr2hno3-nvt.log

 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.246533543175843


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.438732194981611


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.878629961526173


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -143.350064790371448


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -144.026841371125641


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -145.725465772527684


Alternation pulls warnings *or* errors in one pass — the everyday log-triage move:

In [7]:
grep -iE 'warning|error' data/logs/gr2hno3-nvt.log

 *** WARNING in motion/simpar_methods.F:223 :: A temperature tolerance ***


 *** WARNING in dbcsr_mm.F:295 :: Using a non-square number of MPI ranks ***


 The number of warnings for this run is : 2


```{admonition} ERE vs BRE, and what we are skipping
:class: note
Reach for `grep -E`. Without it, grep uses the older **Basic** regex (BRE), in
which `+ ? { } ( ) |` are *literal characters* unless you backslash them — a
reliable source of "why doesn't my pattern work". `-E` gives the Extended syntax
above, and the same `-E` works for `sed` later. Two things we deliberately leave
out as beyond a prerequisite course: Perl-style `grep -P` (PCRE), and
backreferences / lookaround. They exist; you do not need them yet.
```

## C. Pattern and flag together, on a real log

The power is in combining a regex with the flags. The signed-decimal pattern, fed
the energy lines, and then `-o` to print **only the matched part** — the numbers
themselves, stripped of the surrounding text:

In [8]:
grep 'Total FORCE_EVAL' data/logs/gr2hno3-nvt.log | grep -oE '\-[0-9]+\.[0-9]+'

-142.246533543175843


-142.438732194981611


-142.878629961526173


-143.350064790371448


-144.026841371125641


-145.725465772527684


That is a first taste of **extraction** — turning lines into values. Mind the
scope boundary, though: `grep` *selects* (and with `-o`, clips) text; it does not
do columns or arithmetic. When you want "the third field of every line", that is
`awk`, in Notebook 8. Here, `grep` finds; it does not transform.

## D. `grep -r`, and how it differs from `find`

Everything so far searched one file. `-r` searches a whole **tree**, descending
into every file under a directory. "Which logs mention a warning, and how often?"

In [9]:
grep -rc 'WARNING' data/logs

data/logs/gr2hno3-restart.log:2


data/logs/gr2hno3-nvt.log:2


This is the natural place to draw a line you have been circling since Notebook 5.
Both `find` and `grep -r` walk a directory tree, but they look for different
things:

- **`find`** locates **files**, by their name and metadata (`find data -name
  "*.log"`).
- **`grep -r`** locates **text**, by what is *inside* the files (`grep -r 'WARNING'
  data`).

And, being good Unix citizens, they compose: `find` the files you care about, then
`grep` their contents — a pairing you will reach for constantly.

## Exercises

Regular expressions reward practice, so this is a full set: literal `grep` to warm
up, then ERE on the real logs, the course's dedicated *look-it-up* exercise, and a
capstone that mines a log end to end. `grep` only reads; the one exercise that
*saves* its results writes into a fresh `scratch/`.

### Warm-up 1 (worked) — Match, count, locate

Find the total-energy lines in the run log, count them with `-c`, and use `-l` to
see which of the two logs contain them.

In [10]:
cd "$ROOT"

In [11]:
# (solution hidden on the public site)


6


data/logs/gr2hno3-nvt.log


data/logs/gr2hno3-restart.log


In [12]:
check '[ "$(grep -c "Total FORCE_EVAL" data/logs/gr2hno3-nvt.log)" -eq 6 ] && [ "$(grep -l "Total FORCE_EVAL" data/logs/*.log | wc -l)" -eq 2 ]' \
      "six energy lines counted, and both logs contain them"

✓ six energy lines counted, and both logs contain them


### Warm-up 2 (your turn) — Invert

Use `-v` to print the lines that do **not** match. Count the **non-blank** lines of
the log by inverting a pattern that matches blank lines (`'^[[:space:]]*$'` — start,
any spaces, end).

In [13]:
cd "$ROOT"

In [14]:
# (solution hidden on the public site)


571


In [15]:
total=$(grep -c '' data/logs/gr2hno3-nvt.log)
blank=$(grep -c '^[[:space:]]*$' data/logs/gr2hno3-nvt.log)
nonblank=$(grep -cv '^[[:space:]]*$' data/logs/gr2hno3-nvt.log)
check '[ "$nonblank" -eq $((total - blank)) ] && [ "$nonblank" -gt 0 ]' \
      "the inverted count is exactly the non-blank lines"

✓ the inverted count is exactly the non-blank lines


### Applied 1 (your turn) — Anchors and classes

Write ERE patterns (`grep -E`) for two things in
`data/trajectories/lj38-optimization.xyz`: the comment lines, which **start** with
whitespace then `i =` (use `^` and a POSIX class), and count them — there is one
per frame.

In [16]:
cd "$ROOT"

In [17]:
# (solution hidden on the public site)


 i =        1, E =        -0.0231086608


 i =        4, E =        -0.0279588737


 i =        7, E =        -0.0287650150


63


In [18]:
check '[ "$(grep -cE "^[[:space:]]+i =" data/trajectories/lj38-optimization.xyz)" -eq 63 ]' \
      "the anchored pattern matched all 63 frame-comment lines"

✓ the anchored pattern matched all 63 frame-comment lines


### Applied 2 (your turn) — Quantifiers and alternation

Two patterns on the run log: count the lines that are a **warning or an error**
(`-iE 'warning|error'`), and confirm the signed-decimal pattern
`-?[0-9]+\.[0-9]+` appears on the energy lines.

In [19]:
cd "$ROOT"

In [20]:
# (solution hidden on the public site)


3


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.246533543175843


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.438732194981611


 ENERGY| Total FORCE_EVAL ( QS ) energy (a.u.):             -142.878629961526173


In [21]:
check '[ "$(grep -icE "warning|error" data/logs/gr2hno3-nvt.log)" -eq 3 ] && [ "$(grep -cE "energy \(a\.u\.\):[[:space:]]+-?[0-9]+\.[0-9]+" data/logs/gr2hno3-nvt.log)" -eq 6 ]' \
      "three warning/error lines, and the signed-decimal pattern matched all six energy lines"

✓ three warning/error lines, and the signed-decimal pattern matched all six energy lines


### Applied 3 (worked) — Extract the matched part

Pull *just the numbers* out of the energy lines with `-o` — a first taste of
extraction (the full version, with fields, is `awk` in Notebook 8).

In [22]:
cd "$ROOT"

In [23]:
# (solution hidden on the public site)


-142.246533543175843


-142.438732194981611


-142.878629961526173


-143.350064790371448


-144.026841371125641


-145.725465772527684


In [24]:
check '[ "$(grep "Total FORCE_EVAL" data/logs/gr2hno3-nvt.log | grep -oE "\-[0-9]+\.[0-9]+" | wc -l)" -eq 6 ]' \
      "six numbers extracted, one per energy line"

✓ six numbers extracted, one per energy line


### Discovery (your turn) — Find the flag yourself

This is the course's standing skill: needing a flag you were never handed, and
*finding* it. Your task: print **three lines of context around** each `WARNING` in
the log — the warning *and* its neighbours. That flag is not on the card. Consult
`grep --help` or `man grep`, find the context option (`-C`, or `-A`/`-B`), and use
it.

In [25]:
cd "$ROOT"

In [26]:
# (solution hidden on the public site)


 MD| Dump                1000                              gr2hno3_nvt-1.restart


 *** WARNING in motion/simpar_methods.F:223 :: A temperature tolerance ***


 *** (TEMP_TOL) is used during the MD. Due to the velocity rescaling   ***


 *** algorithm jumps may appear in the conserved quantity.             ***


--


  ... [further MD steps elided for the course excerpt] ...


 -------------------------------------------------------------------------------


 *** WARNING in dbcsr_mm.F:295 :: Using a non-square number of MPI ranks ***


In [27]:
check '[ "$(grep -C 3 "WARNING" data/logs/gr2hno3-nvt.log | wc -l)" -gt "$(grep -c "WARNING" data/logs/gr2hno3-nvt.log)" ]' \
      "the context flag printed more than just the matching lines"

✓ the context flag printed more than just the matching lines


### Composite — putting it together (capstone)

Mine the run log end to end and save a small report, using regex, flags, a pipe
(Notebook 4), and redirection (Notebook 4). In `scratch/`: write the **count** of
SCF steps, the list of **energy values**, and the **warnings with context** into
three files.

In [28]:
cd "$ROOT"; rm -rf scratch; mkdir -p scratch

In [29]:
# (solution hidden on the public site)


6


-142.246533543175843


-142.438732194981611


-142.878629961526173


-143.350064790371448


-144.026841371125641


-145.725465772527684


In [30]:
check '[ "$(cat scratch/n_scf.txt)" -eq 6 ] && [ "$(wc -l < scratch/energies.txt)" -eq 6 ] && [ -s scratch/warnings.txt ]' \
      "the report has 6 SCF steps, 6 energy values, and a non-empty warnings file"

✓ the report has 6 SCF steps, 6 energy values, and a non-empty warnings file


### Optional stretch (your turn) — Recurse, and contrast with find

Search **across the tree** with `grep -r`, then reach the same logs via `find`
(Notebook 5) and grep them — two routes to one answer. Here: count the energy lines
in every log under `data/`, recursively, then do it by piping `find` into `grep`.
*(Aside, for Part IV: a big recursive grep is one of the jobs `xargs -P` can run in
parallel.)*

In [31]:
cd "$ROOT"

In [32]:
# (solution hidden on the public site)


data/logs/gr2hno3-restart.log:3


data/logs/gr2hno3-nvt.log:6


data/logs/gr2hno3-restart.log:3


data/logs/gr2hno3-nvt.log:6


In [33]:
check '[ "$(grep -r "Total FORCE_EVAL" data/logs | wc -l)" -eq "$(find data -name "*.log" -exec grep "Total FORCE_EVAL" {} + | wc -l)" ]' \
      "the two routes — grep -r and find | grep — reach the same matches"

✓ the two routes — grep -r and find | grep — reach the same matches


## Outlook

You can now find and pattern-match text — and you have the regular-expression
foundation that the rest of the course leans on. Next (Notebook 7): the **stream
toolkit** — `cut`, `sort`, `uniq`, `wc`, `tr` — the small tools that *reshape* the
lines `grep` selects, turning a pile of matches into counts, columns, and ordered
tables.

```{compendium-new}
```

<div class="bp-banner" style="margin-top:30px;">
  <div class="bp-series">Take this notebook with you</div>
  <div style="font-size:14.5px;line-height:1.55;max-width:66ch;">
    Open a <b>live terminal</b> from the &ldquo;Practice here&rdquo; box in any
    section to run everything yourself — nothing to install. The published
    notebooks ship <b>without worked solutions</b>; if you would like the
    reference solutions — to teach from or to check your own work — get in
    touch: <a href="mailto:hello@ramador.me">hello@ramador.me</a>.
  </div>
</div>